In [1]:
!pip install -q transformers accelerate datasets torch

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="cuda:0"
)
model.eval()
print("loaded on:", model.device)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

loaded on: cuda:0


In [3]:
from harness import generate_natural, extract_pred_answer_with_confidence, is_correct, extract_gsm8k_gold

In [4]:
### The Old Grader

In [5]:
import re

def old_extract_pred_answer(response_text):
    """Pre-fix logic: boxed -> 'answer is X' -> blind last-number. No
    stated-conclusion tier. Kept here only to compare against the fix."""
    if "</think>" not in response_text:
        return None
    after = response_text.split("</think>")[-1]
    m = re.search(r"\\boxed\{([^}]*)\}", after)
    if m:
        return m.group(1).replace(",", "").strip()
    m = re.search(r"answer\s*is\s*\$?([-\d,\.]+)", after, re.IGNORECASE)
    if m:
        return m.group(1).replace(",", "").strip()
    nums = re.findall(r"-?\d[\d,]*\.?\d*", after)
    return nums[-1].replace(",", "") if nums else None

def norm(s):
    if s is None:
        return None
    s = str(s).replace(",", "").replace("$", "").strip()
    try:
        f = float(s)
        return str(int(f)) if f == int(f) else str(f)
    except ValueError:
        return s

In [6]:
### The New Grader

In [7]:
import random, json, time
from datasets import load_dataset

ds = load_dataset("openai/gsm8k", "main", split="test")
idxs = list(range(len(ds)))
random.Random(0).shuffle(idxs)
idxs = idxs[:20]

results = []
disagreements = 0

for rank, i in enumerate(idxs):
    q = ds[i]["question"]
    gold = extract_gsm8k_gold(ds[i]["answer"])

    t0 = time.time()
    text, think_tokens, margins, ent, full = generate_natural(
        model, tokenizer, q, max_new_tokens=4096, seed=0)
    dt = time.time() - t0

    old_pred = norm(old_extract_pred_answer(text))
    new_pred, conf, method = extract_pred_answer_with_confidence(text)

    old_correct = (old_pred == norm(gold))
    new_correct = is_correct(new_pred, gold)
    disagree = (old_pred != new_pred)
    disagreements += int(disagree)

    row = dict(gsm8k_idx=i, gold=gold, old_pred=old_pred, old_correct=old_correct,
               new_pred=new_pred, new_correct=new_correct, method=method,
               confidence=conf, disagree=disagree, think_tokens=think_tokens,
               wall_clock_sec=round(dt, 1), text=text)
    results.append(row)
    print(f"[{rank+1}/20] idx={i} old={old_pred}({old_correct}) "
          f"new={new_pred}({new_correct}) method={method} conf={conf} disagree={disagree}")

with open("grader_test_n20.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"\nTotal disagreements: {disagreements}/20")
print(f"Old accuracy: {sum(r['old_correct'] for r in results)}/20")
print(f"New accuracy: {sum(r['new_correct'] for r in results)}/20")
from collections import Counter
print("Method distribution:", Counter(r['method'] for r in results))

[1/20] idx=1101 old=16(True) new=16(True) method=boxed conf=1.0 disagree=False
[2/20] idx=120 old=240(True) new=240(True) method=boxed conf=1.0 disagree=False
[3/20] idx=474 old=80(True) new=80(True) method=boxed conf=1.0 disagree=False
[4/20] idx=991 old=15(True) new=15(True) method=boxed conf=1.0 disagree=False
[5/20] idx=192 old=91(True) new=91(True) method=boxed conf=1.0 disagree=False
[6/20] idx=1020 old=22(True) new=22(True) method=boxed conf=1.0 disagree=False
[7/20] idx=350 old=8\ \text{hours(False) new=8(True) method=boxed conf=1.0 disagree=True
[8/20] idx=480 old=11(True) new=11(True) method=boxed conf=1.0 disagree=False
[9/20] idx=883 old=25(True) new=25(True) method=boxed conf=1.0 disagree=False
[10/20] idx=1062 old=\482(False) new=482(True) method=boxed conf=1.0 disagree=True
[11/20] idx=799 old=4(True) new=4(True) method=boxed conf=1.0 disagree=False
[12/20] idx=1257 old=14(True) new=14(True) method=boxed conf=1.0 disagree=False
[13/20] idx=1070 old=\16.02(False) new=16.0

In [11]:
# same cell as before, just re-run it now
import random, json, time
from datasets import load_dataset

ds = load_dataset("openai/gsm8k", "main", split="test")
idxs = list(range(len(ds)))
random.Random(1).shuffle(idxs)
idxs = idxs[:20]

results = []
for rank, i in enumerate(idxs):
    q = ds[i]["question"]
    gold = extract_gsm8k_gold(ds[i]["answer"])
    t0 = time.time()
    text, think_tokens, margins, ent, full = generate_natural(
        model, tokenizer, q, max_new_tokens=4096, seed=0)
    dt = time.time() - t0
    pred, conf, method = harness.extract_pred_answer_with_confidence(text)  # via module, no ambiguity
    correct = is_correct(pred, gold)
    row = dict(gsm8k_idx=i, gold=gold, pred=pred, correct=correct, method=method,
               confidence=conf, think_tokens=think_tokens, wall_clock_sec=round(dt, 1), text=text)
    results.append(row)
    print(f"[{rank+1}/20] idx={i} pred={pred} correct={correct} method={method} conf={conf}")

with open("grader_test_n20_seed1_verified.json", "w") as f:
    json.dump(results, f, indent=2)
n_correct = sum(r["correct"] for r in results)
print(f"\nAccuracy: {n_correct}/20 = {n_correct/20:.2f}")

[1/20] idx=773 pred=10 correct=True method=boxed conf=1.0
[2/20] idx=486 pred=6 correct=True method=boxed conf=1.0
[3/20] idx=150 pred=4 correct=True method=boxed conf=1.0
[4/20] idx=532 pred=25 correct=True method=boxed conf=1.0
[5/20] idx=38 pred=\text{Insufficient information to determine the speed} correct=False method=boxed conf=1.0
[6/20] idx=295 pred=370 correct=True method=boxed conf=1.0
[7/20] idx=278 pred=40 correct=True method=boxed conf=1.0
[8/20] idx=587 pred=30 correct=True method=boxed conf=1.0
[9/20] idx=560 pred=45 correct=True method=boxed conf=1.0
[10/20] idx=646 pred=79 correct=True method=boxed conf=1.0
[11/20] idx=781 pred=2 correct=False method=boxed conf=1.0
[12/20] idx=1314 pred=40 correct=False method=boxed conf=1.0
[13/20] idx=1070 pred=16.02 correct=False method=boxed conf=1.0
[14/20] idx=994 pred=2 correct=True method=boxed conf=1.0
[15/20] idx=99 pred=58 correct=True method=boxed conf=1.0
[16/20] idx=939 pred=650 correct=True method=boxed conf=1.0
[17/20] 